### 1. Install Dependencies
We need `chromadb` for vector storage, `sentence-transformers` to generate embeddings, and `pandas` for handling the data.

In [1]:
!pip install chromadb sentence-transformers pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 73.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 54.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 81.6 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: opentelemetry-semantic-conventions
    Found existing installation: opentelemetry-semantic-conventions 0.63b1
    Uninstalling opentelemetry-semantic-conventions-0.63b1:
      Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Attempting uninstall: opentelemetry-sdk
    Found existing installation: opentelemetry-sdk 1.42.1
    Uninstalling opentelemetry-sdk-1.42.1:
      Suc

### 2. Create and Load Sample FAQ Data
Let's create a sample CSV file containing FAQ questions and answers, then load it using Pandas.

In [2]:
import pandas as pd
import json

# Sample FAQ dataset
faq_data = [
    {"question": "How do I reset my password?", "answer": "Go to settings, click 'Reset Password', and follow the link sent to your email.", "category": "Account"},
    {"question": "What is the return policy?", "answer": "You can return any product within 30 days of purchase for a full refund.", "category": "Billing/Shipping"},
    {"question": "How can I contact customer support?", "answer": "You can reach customer support 24/7 via live chat or at support@example.com.", "category": "Support"},
    {"question": "Do you offer international shipping?", "answer": "Yes, we ship to over 50 countries worldwide. Shipping fees vary by destination.", "category": "Billing/Shipping"},
    {"question": "Is my data secure?", "answer": "We use industry-standard end-to-end encryption to keep your personal data secure.", "category": "Security"}
]

# Save as CSV
csv_path = 'faq_data.csv'
pd.DataFrame(faq_data).to_csv(csv_path, index=False)

# Load using Pandas
df = pd.read_csv(csv_path)
display(df)

,question,answer,category
0,How do I reset my password?,"Go to settings, click 'Reset Password', and fo...",Account
1,What is the return policy?,You can return any product within 30 days of p...,Billing/Shipping
2,How can I contact customer support?,You can reach customer support 24/7 via live c...,Support
3,Do you offer international shipping?,"Yes, we ship to over 50 countries worldwide. S...",Billing/Shipping
4,Is my data secure?,We use industry-standard end-to-end encryption...,Security


### 3. Initialize ChromaDB and Embeddings Model
We will use a lightweight pre-trained model (`all-MiniLM-L6-v2`) from `sentence-transformers` to convert text into embeddings.

In [3]:
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize Chroma client (In-memory storage for demonstration)
chroma_client = chromadb.Client()

# Initialize the embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Create a collection for FAQs
collection = chroma_client.get_or_create_collection(name="faq_collection")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### 4. Index the FAQs into ChromaDB
We will construct a combined text representation for each FAQ, generate its vector embedding, and store it in ChromaDB.

In [4]:
ids = []
documents = []
embeddings = []
metadatas = []

for index, row in df.iterrows():
    # Combine question and answer for better semantic coverage
    combined_text = f"Question: {row['question']}\nAnswer: {row['answer']}"

    # Generate vector embedding
    vector = embedding_model.encode(combined_text).tolist()

    ids.append(f"faq_{index}")
    documents.append(combined_text)
    embeddings.append(vector)
    metadatas.append({"category": row["category"], "question": row["question"]})

# Add elements to the collection
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=documents,
    metadatas=metadatas
)

print(f"Successfully indexed {len(ids)} FAQs into ChromaDB!")

Successfully indexed 5 FAQs into ChromaDB!


### 5. Semantic Search Querying
Let's write a search function that accepts a query, embeds it, retrieves the top 3 most relevant results from ChromaDB, and displays them.

In [5]:
def semantic_search(query, n_results=3):
    # Embed the user query
    query_vector = embedding_model.encode(query).tolist()

    # Query ChromaDB
    results = collection.query(
        query_embeddings=[query_vector],
        n_results=n_results
    )

    # Display results nicely
    print(f"\nSearch Query: '{query}'")
    print("="*50)
    for i in range(len(results['ids'][0])):
        print(f"Rank {i+1} (Distance: {results['distances'][0][i]:.4f})")
        print(f"Document:\n{results['documents'][0][i]}")
        print(f"Metadata: {results['metadatas'][0][i]}")
        print("-"*50)

# Try semantic search queries
semantic_search("How can I get a refund for my order?")
semantic_search("Is my account safe?")


Search Query: 'How can I get a refund for my order?'
Rank 1 (Distance: 1.0972)
Document:
Question: What is the return policy?
Answer: You can return any product within 30 days of purchase for a full refund.
Metadata: {'question': 'What is the return policy?', 'category': 'Billing/Shipping'}
--------------------------------------------------
Rank 2 (Distance: 1.2847)
Document:
Question: How can I contact customer support?
Answer: You can reach customer support 24/7 via live chat or at support@example.com.
Metadata: {'question': 'How can I contact customer support?', 'category': 'Support'}
--------------------------------------------------
Rank 3 (Distance: 1.6974)
Document:
Question: How do I reset my password?
Answer: Go to settings, click 'Reset Password', and follow the link sent to your email.
Metadata: {'category': 'Account', 'question': 'How do I reset my password?'}
--------------------------------------------------

Search Query: 'Is my account safe?'
Rank 1 (Distance: 1.1235)
